# 📄 Build a RAG Pipeline from Scratch — LangChain + OpenAI + FAISS

**Video:** *Build a RAG Pipeline with LangChain* | **Channel:** Prashant Nair (@prashantnairofficial)

---

### What We're Building:
A complete RAG (Retrieval-Augmented Generation) pipeline that can answer questions about **India's Union Budget 2025-26** — a 50+ page government PDF.

### The Pipeline:
```
PDF → Load → Chunk → Embed → Store (FAISS) → Query → Retrieve → LLM → Answer
```

### RAG = Open-Book Exam for AI
Instead of relying on what the LLM "knows," we RETRIEVE relevant information from the document, AUGMENT the prompt with that context, and let the LLM GENERATE a grounded answer.

---

**Tech Stack:** LangChain, OpenAI (gpt-4o-mini), FAISS, PyPDF  
**Author:** Prashant Nair | AI & GenAI Practitioner | Principal Trainer

## 📦 Step 1: Install Dependencies

In [1]:
# Install dependencies (watch for errors — don't skip this output!)
!pip install --quiet langchain==0.3.25 langchain-core==0.3.62 langchain-openai==0.3.18 langchain-community==0.3.24 faiss-cpu==1.11.0 pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.4/438.4 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 100.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

## 🔐 Step 2: Configure OpenAI API Key

Store your API key in **Colab Secrets** (🔑 icon in left panel):  
- Key name: `OPENAI_API_KEY`
- Value: Your OpenAI API key (starts with `sk-`)

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("✅ OpenAI API key configured")

✅ OpenAI API key configured


## 📥 Step 3: Download the Union Budget 2025-26 PDF

We'll use the actual Union Budget document from the Government of India website.  
This is a real 50+ page policy document — the perfect test case for RAG.

In [3]:
import urllib.request

# Union Budget 2025-26 — Budget Speech PDF
PDF_URL = "https://www.indiabudget.gov.in/doc/Budget_Speech.pdf"
PDF_PATH = "union_budget_2025_26.pdf"

urllib.request.urlretrieve(PDF_URL, PDF_PATH)

import os
size_mb = os.path.getsize(PDF_PATH) / (1024 * 1024)
print(f"✅ Downloaded: {PDF_PATH} ({size_mb:.1f} MB)")

✅ Downloaded: union_budget_2025_26.pdf (0.6 MB)


## 📖 Step 4: Load the PDF

LangChain's `PyPDFLoader` reads each page as a separate document object with text content and metadata (page number).

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"✅ Loaded {len(pages)} pages from the Union Budget PDF")
print(f"\n📄 Preview of Page 1 (first 500 chars):")
print(pages[0].page_content[:500])
print(f"\n📋 Metadata: {pages[0].metadata}")

✅ Loaded 65 pages from the Union Budget PDF

📄 Preview of Page 1 (first 500 chars):
GOVERNMENT OF INDIA
BUDGET 2026-2027
SPEECH
OF
NIRMALA SITHARAMAN
MINISTER OF FINANCE
February 1,  2026

📋 Metadata: {'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': 'union_budget_2025_26.pdf', 'total_pages': 65, 'page': 0, 'page_label': '1'}


## ✂️ Step 5: Chunk the Document

We split the pages into smaller, overlapping chunks. Why?
- LLMs have context window limits
- Smaller chunks = more precise retrieval
- Overlap ensures we don't cut important sentences in half

**Settings:**
- `chunk_size=1000` — each chunk is ~1000 characters (~200 words)
- `chunk_overlap=200` — consecutive chunks share 200 characters of context

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print(f"✅ Created {len(chunks)} chunks from {len(pages)} pages")
print(f"\n📊 Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")
print(f"\n📝 Sample chunk (chunk #5):")
print(f"   Content: {chunks[5].page_content[:300]}...")
print(f"   Source: Page {chunks[5].metadata.get('page', 'N/A')}")

✅ Created 149 chunks from 65 pages

📊 Average chunk size: 780 characters

📝 Sample chunk (chunk #5):
   Content: global markets, exporting more and attracting s table long -term 
investment. 
 
Part A 
5. As I begin Part A, I want to express my gratitude to the people for 
standing firmly with us as we forge our way together towards becoming 
one of the largest economies of the world. 
6. Our aim is to transfo...
   Source: Page 5


## 🧮 Step 6: Create Embeddings & Build FAISS Vector Store

**Embeddings** convert text into numerical vectors that capture meaning.  
Similar texts → similar vectors → found by similarity search.

Think of it as placing every chunk on a **"meaning map"** — when you ask a question, we find the chunks closest to your question on that map.

**FAISS** (Facebook AI Similarity Search) stores these vectors and searches them blazingly fast.

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Create embeddings using OpenAI's text-embedding-3-small
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Build the FAISS vector store from chunks
print("⏳ Embedding chunks and building FAISS index...")
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"✅ FAISS vector store created!")
print(f"   Chunks indexed: {vectorstore.index.ntotal}")
print(f"   Embedding dimensions: {vectorstore.index.d}")
print(f"\n   Every chunk is now a {vectorstore.index.d}-dimensional vector.")
print(f"   Ready for similarity search!")

⏳ Embedding chunks and building FAISS index...
✅ FAISS vector store created!
   Chunks indexed: 149
   Embedding dimensions: 1536

   Every chunk is now a 1536-dimensional vector.
   Ready for similarity search!


## 🔍 Step 7: Test Retrieval (Before Adding the LLM)

Let's verify the vector store works by doing a raw similarity search.  
This shows you EXACTLY what the retriever will feed to the LLM.

In [7]:
# Raw similarity search — what the retriever sees
test_query = "What are the new income tax slabs?"

results = vectorstore.similarity_search(test_query, k=3)

print(f"🔍 Query: \"{test_query}\"")
print(f"📄 Top {len(results)} retrieved chunks:\n")

for i, doc in enumerate(results, 1):
    print(f"  ── Chunk {i} (Page {doc.metadata.get('page', 'N/A')}) ──")
    print(f"  {doc.page_content[:300]}...")
    print()

🔍 Query: "What are the new income tax slabs?"
📄 Top 3 retrieved chunks:

  ── Chunk 1 (Page 23) ──
  20  
 
PART B 
Direct Taxes 
Speaker Sir,  
98. Now I present my proposals on Direct Taxes. 
New Income Tax Act 
99. In July 2024, I announced a comprehensive review of the 
Income Tax Act, 1961. This was completed in a record time and the 
Income Tax Act, 2025 will come into effect from 1st April, ...

  ── Chunk 2 (Page 40) ──
  fine and with no minimum imprisonment.  
 It is further proposed that prosecution for the offences under 
Income-tax Act, 2025 shall be based on the amount of tax evaded 
and the punishment shall be proportionate to the gravity of crime. 
In such cases, the requirement of maximum punishment of 
imp...

  ── Chunk 3 (Page 28) ──
  shareholders as Capital Gains. However, to disincentivize misuse of 
tax arbitrage, promoters will pay an additional buyback tax. This will 
make effective tax 22 percent for corporate promoters. For non -
corporate promoters the eff

## 🤖 Step 8: Build the RAG Chain

Now we connect everything: **Retriever + LLM + Prompt = RAG Chain**

The chain does three things automatically:
1. Takes your question
2. Retrieves the most relevant chunks from FAISS
3. Sends them + your question to GPT-4o-mini for a grounded answer

In [8]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# Custom prompt that instructs the LLM to only use retrieved context
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions about India's Union Budget 2025-26.
Use ONLY the following context to answer. If the answer is not in the context, say "I couldn't find this information in the Budget document."

Always mention which part of the Budget your answer comes from when possible.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

# Build the RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

print("✅ RAG Chain built!")
print("   Retriever: FAISS (top 4 chunks)")
print("   LLM: GPT-4o-mini")
print("   Chain: RetrievalQA (stuff method)")
print("\n🎯 Ready to answer questions about the Union Budget!")

✅ RAG Chain built!
   Retriever: FAISS (top 4 chunks)
   LLM: GPT-4o-mini
   Chain: RetrievalQA (stuff method)

🎯 Ready to answer questions about the Union Budget!


## 🚀 Step 9: Ask Questions — Live Demo!

Let's put the pipeline to work. We'll ask 4 different questions about the Budget and see how the RAG chain responds with grounded, sourced answers.

In [9]:
def ask_budget(question):
    """Ask a question about the Union Budget and display the answer with sources."""
    print(f"\n{'═' * 70}")
    print(f"❓ QUESTION: {question}")
    print(f"{'═' * 70}")

    result = rag_chain.invoke({"query": question})

    print(f"\n🤖 ANSWER:")
    print(f"{result['result']}")

    print(f"\n📄 SOURCES (retrieved chunks):")
    seen_pages = set()
    for doc in result['source_documents']:
        page = doc.metadata.get('page', 'N/A')
        if page not in seen_pages:
            seen_pages.add(page)
            print(f"   → Page {page}: {doc.page_content[:120]}...")

    return result

In [10]:
# 🔹 Query 1: Tax Slabs
ask_budget("What are the new income tax slabs announced in the Union Budget 2025-26?");


══════════════════════════════════════════════════════════════════════
❓ QUESTION: What are the new income tax slabs announced in the Union Budget 2025-26?
══════════════════════════════════════════════════════════════════════

🤖 ANSWER:
I couldn't find this information in the Budget document.

📄 SOURCES (retrieved chunks):
   → Page 23: 20  
 
PART B 
Direct Taxes 
Speaker Sir,  
98. Now I present my proposals on Direct Taxes. 
New Income Tax Act 
99. In ...
   → Page 40: fine and with no minimum imprisonment.  
 It is further proposed that prosecution for the offences under 
Income-tax Ac...
   → Page 28: shareholders as Capital Gains. However, to disincentivize misuse of 
tax arbitrage, promoters will pay an additional buy...
   → Page 29: 26  
 
143. MAT is proposed to be made final tax. So, there will be no 
further credit accumulation from 1 st April 2026...


In [11]:
# 🔹 Query 2: Startups & Innovation
ask_budget("What measures were announced for startups, innovation, and entrepreneurship?");


══════════════════════════════════════════════════════════════════════
❓ QUESTION: What measures were announced for startups, innovation, and entrepreneurship?
══════════════════════════════════════════════════════════════════════

🤖 ANSWER:
I couldn't find this information in the Budget document.

📄 SOURCES (retrieved chunks):
   → Page 9: that will promote manufacturing, research and innovation in equipment 
design as well as material sciences.  
Rejuvenati...
   → Page 13: 10  
 
of more than ₹1000 crore. The current scheme under AMRUT which 
incentivises issuances up to ₹200 crore, will als...
   → Page 6: growth, I propose interventions in six areas: i) Scaling up manufacturing 
in 7 strategic and frontier sectors; ii) Reju...


In [12]:
# 🔹 Query 3: Agriculture
ask_budget("What are the key announcements for agriculture and farmers in this budget?");


══════════════════════════════════════════════════════════════════════
❓ QUESTION: What are the key announcements for agriculture and farmers in this budget?
══════════════════════════════════════════════════════════════════════

🤖 ANSWER:
The key announcements for agriculture and farmers in the Union Budget 2025-26 include:

1. **Fisheries Development**: Initiatives for integrated development of 500 reservoirs and Amrit Sarovars, strengthening the fisheries value chain in coastal areas, and enabling market linkages involving start-ups and women-led groups along with Fish Farmers Producer Organisations (Part 76).

2. **Animal Husbandry Support**: Introduction of a loan-linked capital subsidy support scheme for establishing veterinary and para-vet colleges, veterinary hospitals, diagnostic laboratories, and breeding facilities in the private sector, with a goal to scale up the availability of veterinary professionals (Part 59).

3. **High Value Agriculture**: Support for high value cro

In [13]:
# 🔹 Query 4: AI & Technology
ask_budget("What did the budget say about artificial intelligence, technology, and digital infrastructure?");


══════════════════════════════════════════════════════════════════════
❓ QUESTION: What did the budget say about artificial intelligence, technology, and digital infrastructure?
══════════════════════════════════════════════════════════════════════

🤖 ANSWER:
The budget emphasizes that the 21st Century is technology-driven and highlights the government's commitment to adopting technology for the benefit of all people, including farmers, women in STEM, youth, and individuals with disabilities (Divyangjan). It mentions several initiatives to support new technologies through the AI Mission and the National Quantum Mission. 

Additionally, the budget proposes measures for upskilling and re-skilling technology professionals and engineers in AI and emerging technologies. It also includes plans for AI-enabled matching of workers, jobs, and training opportunities, as well as steps to make informal workflows visible and future-ready to enhance upward mobility prospects. 

This information can 

In [14]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# Custom prompt that instructs the LLM to only use retrieved context
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions about India's Union Budget 2025-26.
Use the following context to answer. Combine information from multiple context passages if needed.
If you can only find partial information, share what you find and note what might be missing.
Always cite which section or paragraph your answer comes from when possible.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

# Build the RAG chain
rag_chain1 = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 6}),
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

print("✅ RAG Chain built!")
print("   Retriever: FAISS (top 6 chunks)")
print("   LLM: GPT-4o-mini")
print("   Chain: RetrievalQA (stuff method)")
print("\n🎯 Ready to answer questions about the Union Budget!")

✅ RAG Chain built!
   Retriever: FAISS (top 6 chunks)
   LLM: GPT-4o-mini
   Chain: RetrievalQA (stuff method)

🎯 Ready to answer questions about the Union Budget!


In [15]:
def ask_budget(question):
    """Ask a question about the Union Budget and display the answer with sources."""
    print(f"\n{'═' * 70}")
    print(f"❓ QUESTION: {question}")
    print(f"{'═' * 70}")

    result = rag_chain1.invoke({"query": question})

    print(f"\n🤖 ANSWER:")
    print(f"{result['result']}")

    print(f"\n📄 SOURCES (retrieved chunks):")
    seen_pages = set()
    for doc in result['source_documents']:
        page = doc.metadata.get('page', 'N/A')
        if page not in seen_pages:
            seen_pages.add(page)
            print(f"   → Page {page}: {doc.page_content[:120]}...")

    return result

In [16]:
# 🔹 Query 1: Tax Slabs
ask_budget("What are the new income tax slabs announced in the Union Budget 2025-26?");


══════════════════════════════════════════════════════════════════════
❓ QUESTION: What are the new income tax slabs announced in the Union Budget 2025-26?
══════════════════════════════════════════════════════════════════════

🤖 ANSWER:
The provided context does not include specific information regarding the new income tax slabs announced in the Union Budget 2025-26. It mentions a comprehensive review of the Income Tax Act, 1961, leading to the Income Tax Act, 2025, which will come into effect from April 1, 2026, but does not detail the new tax slabs themselves (paragraphs 99-100). 

If you are looking for specific tax slab rates or thresholds, that information is missing from the context provided. You may need to refer to additional sources or sections of the budget for the exact details on the new income tax slabs.

📄 SOURCES (retrieved chunks):
   → Page 23: 20  
 
PART B 
Direct Taxes 
Speaker Sir,  
98. Now I present my proposals on Direct Taxes. 
New Income Tax Act 
99. In ...


In [17]:
# 🔹 Query 2: Startups & Innovation
ask_budget("What measures were announced for startups, innovation, and entrepreneurship?");


══════════════════════════════════════════════════════════════════════
❓ QUESTION: What measures were announced for startups, innovation, and entrepreneurship?
══════════════════════════════════════════════════════════════════════

🤖 ANSWER:
The Union Budget 2025-26 includes several measures aimed at promoting startups, innovation, and entrepreneurship:

1. **Support for MSMEs**: A dedicated ₹10,000 crore SME Growth Fund is proposed to create future "Champion SMEs" by incentivizing enterprises based on select criteria (paragraph 28). Additionally, the Self-Reliant India Fund will be topped up with ₹2,000 crore to continue supporting micro enterprises and maintain their access to risk capital (paragraph 29).

2. **Rejuvenation of Legacy Industrial Clusters**: A scheme is proposed to revive 200 legacy industrial clusters to improve their cost competitiveness and efficiency through infrastructure and technology upgradation (paragraph 26). This initiative aims to enhance the overall indus

In [18]:
# 🔹 Query 3: Agriculture
ask_budget("What are the key announcements for agriculture and farmers in this budget?");


══════════════════════════════════════════════════════════════════════
❓ QUESTION: What are the key announcements for agriculture and farmers in this budget?
══════════════════════════════════════════════════════════════════════

🤖 ANSWER:
The Union Budget 2025-26 includes several key announcements aimed at supporting agriculture and enhancing the livelihoods of farmers:

1. **Support for Small and Marginal Farmers**: The budget emphasizes attention to small and marginal farmers, aiming to empower them through various initiatives (Section 15).

2. **Fisheries Development**: Initiatives will be undertaken for the integrated development of 500 reservoirs and Amrit Sarovars. There will also be efforts to strengthen the fisheries value chain in coastal areas, involving start-ups and women-led groups alongside Fish Farmers Producer Organisations (Paragraph 76).

3. **Animal Husbandry Support**: The budget proposes a loan-linked capital subsidy support scheme to establish veterinary and par

In [19]:
# 🔹 Query 4: AI & Technology
ask_budget("What did the budget say about artificial intelligence, technology, and digital infrastructure?");


══════════════════════════════════════════════════════════════════════
❓ QUESTION: What did the budget say about artificial intelligence, technology, and digital infrastructure?
══════════════════════════════════════════════════════════════════════

🤖 ANSWER:
The Union Budget 2025-26 emphasizes the importance of artificial intelligence (AI) and emerging technologies as pivotal for the 21st century, highlighting their role in benefiting various segments of society, including farmers, women in STEM, and youth seeking to upskill. The government has initiated several measures to support the adoption of these technologies, including the AI Mission and the National Quantum Mission (paragraph 48).

Additionally, the budget proposes specific measures to integrate AI into the education system, including embedding AI in the curriculum from school level onwards and upgrading teacher training institutes (paragraph 32). There are also plans for upskilling and reskilling technology professionals in

## 💬 Step 10: Interactive Mode — Ask Your Own Questions

In [ ]:
# ── Ask your own question about the Union Budget! ──
# Modify the question below and run the cell

your_question = "What is the total expenditure planned for education?"

ask_budget(your_question);


══════════════════════════════════════════════════════════════════════
❓ QUESTION: What is the total expenditure planned for education?
══════════════════════════════════════════════════════════════════════

🤖 ANSWER:
I couldn't find this information in the Budget document.

📄 SOURCES (retrieved chunks):
   → Page 22: 19  
 
estimated at par with BE of 2025 -26 at 4.4 percent of GDP. In line with 
the new fiscal prudence path of debt co...
   → Page 10: FY2014-15 to an allocation of ₹11.2 lakh crore in  
BE 2025-26. In FY2026-27, I propose to increase it to ₹12.2 lakh cro...
   → Page 21: provided ₹1.4 lakh crore to the States for the FY 2026 -27 as Finance 
Commission Grants. These include Rural and Urban ...


## 📊 Step 11: Pipeline Summary — What We Built

In [20]:
print(f"""
╔══════════════════════════════════════════════════════════════╗
║         RAG PIPELINE — COMPLETE SUMMARY                    ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  📄 SOURCE:     Union Budget 2025-26 ({len(pages)} pages)          ║
║  ✂️  CHUNKS:     {len(chunks)} chunks (1000 chars, 200 overlap)      ║
║  🧮 EMBEDDINGS: OpenAI text-embedding-3-small             ║
║  💾 VECTOR DB:  FAISS ({vectorstore.index.ntotal} vectors, {vectorstore.index.d}D)         ║
║  🤖 LLM:        GPT-4o-mini (temperature=0.2)             ║
║  🔗 CHAIN:      RetrievalQA (top-4 retrieval)             ║
║                                                            ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  📐 The Pipeline:                                          ║
║  PDF → Load → Chunk → Embed → FAISS → Retrieve → LLM     ║
║                                                            ║
║  🔑 KEY INSIGHT:                                           ║
║  "RAG is an open-book exam for AI."                        ║
║                                                            ║
║  🔗 CONNECTION TO AGENTIC AI:                              ║
║  RAG is the MEMORY layer of Agentic systems.               ║
║  When an AI agent needs to look something up,              ║
║  this is the mechanism it uses.                            ║
║                                                            ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  🔜 LIMITATIONS OF BASIC RAG:                              ║
║  → What if retrieval pulls wrong chunks?                   ║
║  → What about multi-document search?                       ║
║  → What about conversation history?                        ║
║                                                            ║
║  Solutions: Corrective RAG, Self-RAG, Agentic RAG          ║
║  → Coming in future videos!                                ║
║                                                            ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  📺 SUBSCRIBE: @prashantnairofficial                       ║
║  New videos every week on GenAI, RAG, and Agentic AI.      ║
║                                                            ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║         RAG PIPELINE — COMPLETE SUMMARY                    ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  📄 SOURCE:     Union Budget 2025-26 (65 pages)          ║
║  ✂️  CHUNKS:     149 chunks (1000 chars, 200 overlap)      ║
║  🧮 EMBEDDINGS: OpenAI text-embedding-3-small             ║
║  💾 VECTOR DB:  FAISS (149 vectors, 1536D)         ║
║  🤖 LLM:        GPT-4o-mini (temperature=0.2)             ║
║  🔗 CHAIN:      RetrievalQA (top-4 retrieval)             ║
║                                                            ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  📐 The Pipeline:                                          ║
║  PDF → Load → Chunk → Embed → FAISS → Retrieve → LLM     ║
║                                                            ║
║  🔑 K

## 🧪 Bonus: Use Your Own PDF

Want to try this with your own document? Upload any PDF to Colab and change the path below.

In [ ]:
# ── Upload your own PDF and test! ──
# 1. Click the folder icon (📁) on the left panel in Colab
# 2. Upload your PDF
# 3. Change the path below
# 4. Run this cell

YOUR_PDF = "union_budget_2025_26.pdf"  # ← Change this to your uploaded file

# Rebuild the pipeline with your document
custom_loader = PyPDFLoader(YOUR_PDF)
custom_pages = custom_loader.load()
custom_chunks = text_splitter.split_documents(custom_pages)
custom_vectorstore = FAISS.from_documents(custom_chunks, embeddings)

custom_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=custom_vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
)

print(f"✅ New RAG pipeline built for: {YOUR_PDF}")
print(f"   Pages: {len(custom_pages)} | Chunks: {len(custom_chunks)}")

# Ask a question about YOUR document
your_q = "What is this document about?"
result = custom_chain.invoke({"query": your_q})
print(f"\n❓ {your_q}")
print(f"🤖 {result['result']}")

---

## 🔜 What's Next?

| Video | Topic | Status |
|-------|-------|--------|
| #1 | What is Agentic AI? | ✅ Done |
| #2 | Build a RAG Pipeline (This video) | ✅ Done |
| #3 | LangChain vs LangGraph — When to Use What | 🔜 Next |
| #4 | Agentic Design Patterns Deep Dive | 📋 Planned |
| #5 | Advanced RAG: Corrective RAG & Agentic RAG | 📋 Planned |

**Subscribe** so you don't miss them: [Prashant Nair on YouTube](https://youtube.com/@prashantnairofficial)

---

*Built by Prashant Nair | AI & GenAI Practitioner | Principal Trainer*  
*Tech: LangChain + OpenAI (gpt-4o-mini) + FAISS*